In [5]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import pandas as pd

run1 = Path("/kaggle/input/datasets/mrsingh710/results/results_before_discontinuation/results_before_discontinuation/runs/yolo11s_indian_food")
run2 = Path("/kaggle/input/datasets/mrsingh710/results/results_after_discontinuation/results_after_discontinuation/runs/yolo11s_indian_food")
outdir = Path("output/yolo_all_outputs")
outdir.mkdir(parents=True, exist_ok=True)

csv1 = run1 / "results.csv"
csv2 = run2 / "results.csv"

df1 = pd.read_csv(csv1)
df2 = pd.read_csv(csv2)

df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

df1["epoch"] = range(1, len(df1) + 1)
df2["epoch"] = range(len(df1) + 1, len(df1) + len(df2) + 1)

df = pd.concat([df1, df2], ignore_index=True)
df.to_csv(outdir / "merged_results_1_to_100.csv", index=False)

print("Merged CSV saved to:", outdir / "merged_results_1_to_100.csv")
print(df.columns.tolist())

Merged CSV saved to: output/yolo_all_outputs/merged_results_1_to_100.csv
['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2', 'lr/pg3', 'lr/pg4', 'lr/pg5', 'lr/pg6', 'lr/pg7']


In [9]:
from pathlib import Path
import pandas as pd

csv_path = Path("output/yolo_all_outputs/merged_results_1_to_100.csv")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

if "epoch" not in df.columns:
    df["epoch"] = range(1, len(df) + 1)

print(df.columns.tolist())
print(df.head())

['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2', 'lr/pg3', 'lr/pg4', 'lr/pg5', 'lr/pg6', 'lr/pg7']
   epoch     time  train/box_loss  train/cls_loss  train/dfl_loss  \
0      1  1011.34         1.11398         2.39729         1.50858   
1      2  1836.14         1.12419         1.86527         1.49176   
2      3  2614.69         1.18654         2.07200         1.54502   
3      4  3376.69         1.17467         2.00281         1.53483   
4      5  4140.91         1.10760         1.75618         1.47544   

   metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
0               0.48098            0.43691           0.41831   
1               0.43960            0.45076           0.40213   
2               0.47047            0.44109           0.40762   
3               0.52440            0.5518

In [10]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

outdir = Path("output/yolo_all_outputs")
outdir.mkdir(parents=True, exist_ok=True)

plot_cols = [
    "train/box_loss", "train/cls_loss", "train/dfl_loss",
    "metrics/precision(B)", "metrics/recall(B)",
    "val/box_loss", "val/cls_loss", "val/dfl_loss",
    "metrics/mAP50(B)", "metrics/mAP50-95(B)"
]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for ax, col in zip(axes, plot_cols):
    if col not in df.columns:
        ax.set_visible(False)
        continue

    x = df["epoch"].to_numpy()
    y = df[col].to_numpy(dtype=float)
    smooth = gaussian_filter1d(y, sigma=2)

    ax.plot(x, y, marker="o", linewidth=2.5, label="results")
    ax.plot(x, smooth, linestyle=":", linewidth=2.5, label="smooth")
    ax.set_title(col, fontsize=14)
    ax.set_xlim(1, 100)
    ax.grid(True, alpha=0.25)

    if col in [
        "train/box_loss", "train/cls_loss", "train/dfl_loss",
        "val/box_loss", "val/cls_loss", "val/dfl_loss"
    ]:
        ax.set_ylim(bottom=min(y) * 0.98, top=max(y) * 1.02)
    else:
        ax.set_ylim(bottom=min(y) * 0.995, top=max(y) * 1.005)

    if col == "train/cls_loss":
        ax.legend()

plt.tight_layout()
plt.savefig(outdir / "ultralytics_style_epoch_curves.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", outdir / "ultralytics_style_epoch_curves.png")

Saved: output/yolo_all_outputs/ultralytics_style_epoch_curves.png


In [16]:
from ultralytics import YOLO
from pathlib import Path
import shutil

model_path = "/kaggle/input/datasets/mrsingh710/results/results_before_discontinuation/results_before_discontinuation/runs/yolo11s_indian_food/weights/best.pt"
data_yaml = "/kaggle/input/datasets/mrsingh710/results/results_before_discontinuation/results_before_discontinuation/data.yaml"
outdir = Path("output/yolo_all_outputs")
outdir.mkdir(parents=True, exist_ok=True)

model = YOLO(model_path)

results = model.val(
    data=data_yaml,
    imgsz=640,
    conf=0.25,
    iou=0.45,
    plots=True,
    save_json=True,
    split="val"
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)
print("Precision:", results.box.mp)
print("Recall:", results.box.mr)

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,440,664 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 111.7±70.3 MB/s, size: 63.6 KB)
val: Scanning /kaggle/input/datasets/mrsingh710/food-dataset/food_dataset/valid/labels... 5922 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5922/5922 693.6it/s 8.5s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/mrsingh710/food-dataset/food_dataset/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 371/371 4.4it/s 1:25<0.2ss
                   all       5922       8812      0.825      0.806      0.776      0.637
             aloo_gobi        119        121      0.959      0.959      0.958      0.868
           aloo_masala        111        115      0.944       0.93      0.927      0.839
                 appam         45       

In [15]:
from ultralytics import YOLO
from pathlib import Path
import shutil

model_path = "/kaggle/input/datasets/mrsingh710/results/results_before_discontinuation/results_before_discontinuation/runs/yolo11s_indian_food/weights/best.pt"
data_yaml = "/kaggle/input/datasets/mrsingh710/results/results_before_discontinuation/results_before_discontinuation/data.yaml"
outdir = Path("output/yolo_all_outputs")
outdir.mkdir(parents=True, exist_ok=True)

model = YOLO(model_path)

results = model.val(
    data=data_yaml,
    imgsz=640,
    conf=0.25,
    iou=0.45,
    plots=True,
    save_json=True,
    split="test"
)

print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)
print("Precision:", results.box.mp)
print("Recall:", results.box.mr)

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,440,664 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 213.6±329.1 MB/s, size: 124.5 KB)
val: Scanning /kaggle/input/datasets/mrsingh710/food-dataset/food_dataset/test/labels... 5919 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5919/5919 627.2it/s 9.4s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/mrsingh710/food-dataset/food_dataset/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 370/370 4.4it/s 1:25<0.2ss
                   all       5919       8875      0.838      0.795      0.765      0.625
             aloo_gobi        119        120      0.977      0.975      0.975       0.86
           aloo_masala        112        113      0.981      0.921      0.934      0.838
                 appam         45        